COSTRUIRE FRONTEND AI CON HTML CSS E JS + API AI BACKEND

Per l'utente finale l'ai è un entità astratta, iniziamo a dare un volto a questa intelligenza, in modo che anche l'utente finale possa guidarla

Come far parlare una pagine web leggere con un cervello ai pesante.

- Architettura disaccopiata: Axios che chiama l'API LLM Studio
- Gestione stati asincroni: pazienda digitale e gli errori, per non lascia l'utente nel dubbio mentre il modello sta pensando
- Rendering Markdown: visualizzare correttamente le risposte formattate degli LLM

Come separiamo il corpo dell'applicazione dal suo cervello?

Perchè separiamo frontend e backend
In una pipeline moderna, il frontend agisce come un guscio leggero che non possiede logica di calcolo. La separazione permette di aggiornare il modello AI nel backeend senza dover toccare il codice della UI
Utilizziamo LM Studio come provider locale di API (compatibile OpenAI) e Axios come libreria Javascript per gestire lo scambio di dati via HTTP

Perchè scegliamo Axios invece dello strumento standard dei browser?
Comunicazione via Axios
Perchè non usare la Fetch API nativa
Axios è come un corriere privato di lusso rispetto alla risposta primaria, converte automaticamente i dati JSON in oggetti JS (JavaScript), semplificando la manipolazione del payload.
Possiamo poi configurare gli header ed i token di sicurezza in modo centrallizzato ed grazie agli Interceptors possiamo monitorare ogni pacchetto che entra ed esce per capire se ci sono colli di bottiglia.
Ricorda che il tempo che l'utente aspetta è la somma del viaggio dei dati sui cavi di rete e del tempo di generazione dei token (inference time)

Come queto corriere interagisce specificatamente con il nostro server ai locale
LM Studio è uno strumento che espone un server locale che parla la stessa lingua di openAI, emula l'API di OpenAI sull'endpoint v1/chat/completions, rendendo il frontedn portabile verso altri modelli.
Questo significa che il frontend che scriviamo oggi, funzionerà anche domani se decidiamo di spostarci su modelli professionali in cloud
Dobbiamo strare attenti al CORS un poliziotto del browser che impedisce al sito di parlare con un server sconosciuto. Dobbiamo istruire il nostro server affinchè dia il ben venuto alle richieste provenienti dal nostro frontend.
L'oggetto 'messages' deve essere inviato con ruoli 'system' e 'user' e 'assistant' per mantenere il contesto

Ma quanta informazione possiamo inviare, prima che il sistema inizi a rallentare?

Efficienza delle Chiamate
L'efficienza è tutto, se diamo, ogni volta, tutta la storia della chat, il pacchetto dati diventa piano piano gigantesco, saturando la nostra rete.
La dimensione del payload 'P' cresce linearmente con il numero di messaggi 'N' inclusi nella finestra di contesto.
Dobbiamo imparare a tagliare i ricordi troppo vecchi per mantenere l'applcazione leggere e scattante.
Ingegnerizzare la comunicazione, significa bilanciare quanto l'ai deve ricordare con quanto velocemente vogliamo che risponda

Gestire l'attesa del calcolo
Gestione degli Stati Asincroni
Feedback utente e gestione errori
Le chiamate AI sono intrinsecamente lente. 
Generare una risposta complessa può richedere facilmente secondi o perfino minuti.
Un utente che clicca su un tasto e non vede nulla per 5 secondi, pensa che l'app sia rotta.
Un frontend di qualità deve comunicare all'utente cosa sta succedendo (loading) e gestire i fallimenti con grazie.
Un sistema robusto è implementato su blocchi try-catch per catturare errori di rete o timeout dell'inferenza locale.
Se il server crash non vogliamo che la pagina diventi bianca, vogliamo un messaggio gentile che spieghi cosa è successo.

Ciclo di Vita della Richiesta
Gestire l'attesa
Dobbiamo gestire 3 stati
- Loading State (stato pending): attivazione di uno spinner o di un messaggio di attesa non appena Axios avvia la richiesta POST. qui mostriamo uno spinner o un messaggio di attesa. Se tutta va bene passiamo allo stato di successo
- Error Catching (stato errore): intercettazione di status code come 404 (modello non caricato) o 500 (crash dell'inferenza) tramite l'oggetto error.response. La probabilità di un errore non è mai zero, dipende da quanta memoria il nostro server ha ancora libera, da quanti utenti stanno interrogando il server e da eventi non prevedibili e di altri fattori
- Timeout Management: definizione di un tempo massimo di attesa per evitare che l'interfaccia rimanga bloccata indefinitivamente

Come trasformiamo un fallimento tecnico in un esperienza accettabile per l'utente?
UX dei Fallimenti
La Graceful Degradation è l'arte di fallire bene. Se l'API fallisce, l'app non deve crashare ma mostrare un pulsante 'Riprova' o un suggerimento d'uso semplificato.
L'uso di allert visuali o messaggi nel log (user notifications) per spiegare all'utente se l'errore è dovuto a un prompt troppo lungo o alla mancanza di connessione.
Disabilitare il tasto 'invia' (input locking) mentre la richiesta è in corso, per evitare chiamate multiple accidentali che sovraccaricano il backend

Significato dei codici in modo da poter gestire tutti gli errori in base a ciò che il server ci restituisce
Codice 422 spesso indica che abbiamo inviato dei token che il modello non capisce, o che abbiamo violato la sua finestra di contesto
Un'architettura resiliente deve gestire i casi in cui la risposta non è un JSON valido a causa di interruzione nel flusso dei token.
Se il serve è occupato non riproviamo subito, aspettiamo un secondo poi 2 poi 4 ecc, questo da il tempo al backedn di terminare i calcoli precedenti e liberare la GPU per noi.

Ora che abbiamo i dati, vediamo come renderli bellissimi per l'utente

Rendering e Visualizzazione
Markdown e Formattazione
Gli LLM generano spesso testo strutturato: elenchi puntati, blocchi di codice o tabelle. Visualizzare questo output come testo semplice renderebbe l'app povera e poco leggibile
Dobbiamo renderizzare l'output trasformando la stringa grezza in una stringa comprensibile
Le librerie Javascript permettono di trasformare le stringhe Markdown in HTML formattato in tempo reale.

Come funziona tecnicamente questo processo di traduzione
Markdown Parser
Dalla stringa alla UI ricca
Utilizziamo un Marked.js, utilizzo di un parser leggero per convertire i simboli Markdown (es. #, *, -) in tag HTML corrispondenti
Non basta mostrare il codice, vogliamo che sia colorato e leggibile, per questo integriamo Highlight.js per colorare il codice generato dal modello rendendolo leggibile.
La funzione di renderin trasforma la materia grezza in un albero strutturato
Fondamentale è la Sanitization, dobbiamo assicurarci che l'ai non ci invia codice malevolo che potrebbe essere eseguito nel browser dell'utente.

Il testo però non è solo forma è anche movimento e interazione

UX del Testo Generato
Anche senza streaming dei token, posssiamo applicare effetti di transizione per far apparire il tsto in modo naturale (streaming simulation).
Quando la risposta del modello supera l'altezza dell'area visualizzabile (risposta molto lunga), il Container Scrolling permette la gestione automatica dello scroll verso il basso.
E per i colleghi scienziati includiamo latex per visualizzare le formule matematiche che l'ai potrebbe generare.

Tutta queta libertà di rendering nasconde un pericolo insidioso

Parliamo di sicurezza
L'AI è imprevedibile, tramite un prompt injection un utente malintenzionato potrebbe convincere il modello ha generare involontariamente script malevoli.
Se lo renderizziamo senza protezione, quello script verrà eseguito.
L'uso di 'DOMPurify' garantisce che solo i tag HTML sicuri vengano renderizzati nel browser dell'utente finale. Questo strumento analizza l'output e rimuove tutto ciò che non è un tag innocuo